# RetroSpec v3 - Notebook 1: correctness gate + calibration

**Run this before the benchmark. Every time you change the engine.**

v2's numbers were bad for a reason that no amount of data would have fixed: the
speculative decoder was not lossless. On a mismatch it kept the *rejected*
draft token's KV entry in the cache and then read the next-token distribution
from the wrong position, so the model was conditioning on tokens it had never
emitted. Replaying that bookkeeping against a deterministic mock LM, **60 out
of 60 runs drifted away from greedy decoding**.

It was invisible because quality was scored with ROUGE-L, which degrades
gracefully. Here quality is scored on **token ids**: a lossless method must be
bit-exact or it fails.

This notebook does two things:
1. **Gate** - assert every lossless method reproduces greedy output exactly.
2. **Calibrate** - sweep the draft length gamma per drafter and pick the winner
   on measured latency, not on a guess.


In [ ]:
# --- dependencies -------------------------------------------------------
!pip install -q -U "transformers>=4.45" accelerate bitsandbytes rouge-score 2>&1 | tail -1
# faiss is optional; the hybrid drafter falls back to numpy, which is faster
# than faiss at these index sizes anyway.


In [ ]:
import os, sys, json, glob

# Point this at wherever the `retrospec` package lives.
CANDIDATES = ["/kaggle/working/RetroSpecV2", "/kaggle/input/retrospec", ".", ".."]
REPO = next((p for p in CANDIDATES if os.path.isdir(os.path.join(p, "retrospec"))), None)
if REPO is None:
    REPO = "/kaggle/working/RetroSpecV2"
    os.system(f"git clone -q https://github.com/lxzy8/RetroSpecV2.git {REPO}")
sys.path.insert(0, REPO)
os.chdir(REPO)

from retrospec import bench, data
from retrospec.engine import TorchVerifier, speculative_generate, greedy_generate
from retrospec.drafters import NGramDrafter, HybridDrafter, ModelDrafter, EarlyExitDrafter
from retrospec.calm import calm_generate

print("repo:", REPO)
print(bench.describe_gpu())


In [ ]:
# Llama-3.2 is gated. Store your token as a Kaggle Secret named HF_TOKEN
# (Add-ons -> Secrets), or paste it below.
try:
    from kaggle_secrets import UserSecretsClient
    tok_str = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login; login(tok_str)
    print("logged in to Hugging Face")
except Exception as e:
    print("No HF_TOKEN secret found:", e)
    print("Either add the secret, or switch VERIFIER/DRAFTER below to an "
          "ungated pair such as Qwen/Qwen2.5-1.5B-Instruct + Qwen/Qwen2.5-0.5B-Instruct.")


## Load the verifier

In [ ]:
VERIFIER_NAME = "meta-llama/Llama-3.2-3B-Instruct"
DRAFTER_NAME  = "meta-llama/Llama-3.2-1B-Instruct"   # same tokenizer family -- required
DEVICE        = "cuda:0"                             # pin to ONE gpu, never device_map="auto"
MAX_NEW       = 128                                  # 30 was far too short to measure anything

import torch
DTYPE = bench.pick_dtype()
print("dtype:", DTYPE, "| bf16 hardware:", bench.describe_gpu().get("bf16"))

verifier_model, tok = bench.load_model(VERIFIER_NAME, device=DEVICE, dtype=DTYPE)
V   = TorchVerifier(verifier_model, device=DEVICE)
EOS = bench.eos_ids_for(tok, verifier_model)
SYNC = bench.sync_fn()
print("eos ids:", EOS, "| layers:", len(verifier_model.model.layers))


## 1. Correctness gate

Three short prompts, every drafter, compared token-for-token against greedy
decoding through the exact same engine.

In [ ]:
PROBE = [
    {"id": 0, "prompt": "Explain how a hash table works.", "system_prompt": "You are a helpful assistant."},
    {"id": 1, "prompt": "Write a Python function that reverses a linked list.", "system_prompt": "You are a software engineer."},
    {"id": 2, "prompt": "Summarise the causes of the 1929 Wall Street crash.", "system_prompt": "You are a historian."},
]
PROBE_IDS = [bench.encode(tok, p["prompt"], p["system_prompt"]) for p in PROBE]

def greedy_ref(ids):
    return greedy_generate(V, ids, max_new_tokens=64, eos_ids=EOS, sync=SYNC)

refs = [greedy_ref(i).token_ids for i in PROBE_IDS]
print("reference lengths:", [len(r) for r in refs])
print(tok.decode(refs[0][:60], skip_special_tokens=True))


In [ ]:
def gate(name, make_drafter, need_hidden=False, on_accept_factory=None):
    ok = True
    for ids, ref in zip(PROBE_IDS, refs):
        d = make_drafter()
        cb = on_accept_factory(d) if on_accept_factory else None
        r = speculative_generate(V, ids, draft_fn=d, max_new_tokens=64, eos_ids=EOS,
                                 on_accept=cb, need_hidden=need_hidden, sync=SYNC)
        if r.token_ids != ref:
            ok = False
            n = next((k for k,(a,b) in enumerate(zip(r.token_ids, ref)) if a!=b), min(len(r.token_ids),len(ref)))
            print(f"  FAIL {name}: diverges at token {n}")
            print("   got :", tok.decode(r.token_ids[max(0,n-5):n+8], skip_special_tokens=True))
            print("   want:", tok.decode(ref[max(0,n-5):n+8], skip_special_tokens=True))
    print(("  PASS " if ok else "  FAIL ") + name)
    return ok

results = {}
results["ngram"]  = gate("ngram",  lambda: NGramDrafter(draft_len=5))
results["hybrid"] = gate("hybrid", lambda: HybridDrafter(draft_len=5), need_hidden=True,
                         on_accept_factory=lambda d: (lambda toks, hid: d.add(toks, hid)))


In [ ]:
drafter_model, _ = bench.load_model(DRAFTER_NAME, device=DEVICE, dtype=DTYPE)
results["spec_1b"] = gate("spec_1b", lambda: ModelDrafter(drafter_model, draft_len=5, device=DEVICE))

try:
    results["earlyexit"] = gate("earlyexit",
        lambda: EarlyExitDrafter(verifier_model, n_layers=len(verifier_model.model.layers)//2,
                                 draft_len=4, device=DEVICE))
except Exception as e:
    results["earlyexit"] = None
    print("  SKIP earlyexit:", type(e).__name__, e)

assert all(v for v in results.values() if v is not None), "A lossless method is not bit-exact -- stop and fix the engine before benchmarking."
print("\nAll lossless methods are bit-exact.")


## 2. Calibration

Bigger gamma means fewer verifier forwards but more wasted draft work. The
optimum depends on drafter cost, acceptance rate and your GPU. v2 hard-coded
gamma=4 with no measurement at all. Measure it.

In [ ]:
import time, statistics

def timed(make_drafter, gamma, need_hidden=False, cb_factory=None, reps=2):
    lat, tpf, acc = [], [], []
    for ids in PROBE_IDS:
        for _ in range(reps):
            d = make_drafter(gamma)
            cb = cb_factory(d) if cb_factory else None
            r = speculative_generate(V, ids, draft_fn=d, max_new_tokens=MAX_NEW, eos_ids=EOS,
                                     on_accept=cb, need_hidden=need_hidden, sync=SYNC)
            lat.append(r.latency_sec / max(r.n_tokens, 1))
            tpf.append(r.tokens_per_forward); acc.append(r.acceptance_rate)
    return statistics.median(lat), statistics.mean(tpf), statistics.mean(acc)

# baseline cost per token
base = statistics.median(
    greedy_generate(V, ids, max_new_tokens=MAX_NEW, eos_ids=EOS, sync=SYNC).latency_sec / MAX_NEW
    for ids in PROBE_IDS for _ in range(2))
print(f"baseline: {base*1000:.2f} ms/token\n")

SWEEPS = {
  "ngram":   (lambda g: NGramDrafter(draft_len=g), False, None, [2,3,4,6,8,10]),
  "hybrid":  (lambda g: HybridDrafter(draft_len=g), True,
              (lambda d: (lambda t,h: d.add(t,h))), [2,3,4,6,8]),
  "spec_1b": (lambda g: ModelDrafter(drafter_model, draft_len=g, device=DEVICE), False, None, [2,3,4,5,6,8]),
}

best = {}
for name, (mk, nh, cbf, gammas) in SWEEPS.items():
    print(f"{name}:")
    rows = []
    for g in gammas:
        ms, tpf, acc = timed(mk, g, nh, cbf)
        rows.append((g, ms, tpf, acc))
        print(f"   gamma={g:<3} {ms*1000:6.2f} ms/tok  speedup={base/ms:5.2f}x  tok/fwd={tpf:4.2f}  accept={acc:.2f}")
    g_star = min(rows, key=lambda r: r[1])[0]
    best[name] = g_star
    print(f"   -> gamma* = {g_star}\n")

os.makedirs("configs", exist_ok=True)
cfg = {"gamma": best, "max_new_tokens": MAX_NEW, "dtype": str(DTYPE),
       "verifier": VERIFIER_NAME, "drafter": DRAFTER_NAME, "gpu": bench.describe_gpu()}
json.dump(cfg, open("configs/calibration_v3.json","w"), indent=2)
print(json.dumps(cfg, indent=2))


### Reading the sweep

- **`tok/fwd` is the ceiling.** It is how many tokens each verifier forward
  buys you. If `tok/fwd` is 1.0 the drafter is contributing nothing.
- **speedup well below `tok/fwd`** means drafting overhead is eating the win.
  For `ngram`/`hybrid` that should barely happen now that the indices are
  incremental; if it does, lower gamma.
- **`accept` below ~0.25** means the drafter is wrong more often than not and
  will not pay for itself at any gamma.
